# Partial Derivatives & Gradients

## What's covered

- From one variable to many — **partial derivatives** `∂f/∂x_i`
- The **gradient vector** `∇f` and what its direction and magnitude mean
- **Directional derivative** — slope along any chosen direction
- **Steepest ascent** — why gradient descent uses `-∇f`
- **Level sets** and the gradient's orthogonality to them
- A worked example — the gradient of the **MSE loss** of linear regression
- Where this appears in ML — loss landscapes, gradient descent, every training step


## From one variable to many

In ML, parameters come in fleets. A linear regression has `d` weights and one bias. A small CNN has hundreds of thousands. A modern LLM has hundreds of billions. The loss `L` is one scalar number, but it depends on *all* of them at once: `L(w_1, w_2, ..., w_n)`. Calling it `L(\mathbf{w})` keeps the bookkeeping sane.

Single-variable calculus asked one question: "If I nudge `x`, how does `f(x)` change?" The multivariable version asks two:

- *"If I nudge **one** weight `w_i`, how does `L` change?"* — that's a **partial derivative**.
- *"If I nudge **all** the weights along some direction `u`, how does `L` change?"* — that's a **directional derivative**.

The **gradient** packages all the partial derivatives into a single vector, and that vector secretly knows the answer to the directional question for every direction at once. That's the whole story of this notebook.


## Partial derivatives

The partial derivative of `f(x_1, x_2, ..., x_n)` with respect to `x_i` is what you get by treating all the *other* variables as constants and differentiating like in notebook 2:

$$
\frac{\partial f}{\partial x_i}(\mathbf{x}) = \lim_{h \to 0} \frac{f(x_1, \dots, x_i + h, \dots, x_n) - f(x_1, \dots, x_n)}{h}
$$

That curly `∂` distinguishes "I'm varying only one of several variables" from `d` ("I'm varying *the* variable"). Mechanically, the rules from notebook 2 still apply — treat everything except `x_i` as a constant.

**Example.** `f(x, y) = x^2 + 2y^2 + xy`. Partial with respect to `x`: treat `y` as constant, differentiate:

$$
\frac{\partial f}{\partial x} = 2x + y
$$

Partial with respect to `y`: treat `x` as constant:

$$
\frac{\partial f}{\partial y} = 4y + x
$$

**Notation flavors.** `∂f/∂x_i`, `f_{x_i}`, `D_i f`, `\partial_i f` — all the same thing. In hand-written math you'll often see `f_x` as shorthand for the partial with respect to `x`.

**Numerical version.** Same finite differences as notebook 2, but nudging only one coordinate at a time. We'll lean on this for verification.


In [ ]:
import numpy as np

# Our canonical multivariable function for this notebook
def f(x, y):
    return x ** 2 + 2 * y ** 2 + x * y

# Analytical partials
def df_dx(x, y): return 2 * x + y
def df_dy(x, y): return 4 * y + x

# Numerical partial — central difference along one axis
def partial(f, x, y, axis, h=1e-5):
    if axis == 'x':
        return (f(x + h, y) - f(x - h, y)) / (2 * h)
    else:
        return (f(x, y + h) - f(x, y - h)) / (2 * h)

print(f"{'point':<14} {'∂f/∂x analytic':>18} {'numerical':>14} {'∂f/∂y analytic':>20} {'numerical':>14}")
for x, y in [(1.0, 1.0), (2.0, -1.0), (0.5, 3.0), (-2.0, 0.0)]:
    print(f"({x:>4}, {y:>4})    {df_dx(x, y):>14.4f}     {partial(f, x, y, 'x'):>14.4f}"
          f"     {df_dy(x, y):>14.4f}      {partial(f, x, y, 'y'):>14.4f}")


## The gradient vector

Bundle the partial derivatives into a single column vector and you get the **gradient**:

$$
\nabla f(\mathbf{x}) = \begin{bmatrix} \partial f / \partial x_1 \\ \partial f / \partial x_2 \\ \vdots \\ \partial f / \partial x_n \end{bmatrix}
$$

`∇` (called *nabla* or *del*) is the symbol; `∇f` is a *vector-valued function* — at each point `x` in `R^n` it returns a vector in `R^n`. For our `f(x, y) = x^2 + 2y^2 + xy`:

$$
\nabla f(x, y) = \begin{bmatrix} 2x + y \\ 4y + x \end{bmatrix}
$$

At the point `(1, 1)` the gradient is `(3, 5)`.

**Two readings.**

- **Algebraically**, the gradient is a list of slopes — one for each coordinate axis. It tells you how `f` changes if you wiggle along each axis independently.
- **Geometrically**, the gradient is an *arrow at every point* — its direction points uphill the steepest way, and its length tells you how steep that climb is. We prove this in the next section.

This second reading is where all of ML optimization sits.


In [ ]:
def grad_f(x, y):
    return np.array([df_dx(x, y), df_dy(x, y)])

# Build a grid and compute the gradient at every point
xs = np.linspace(-2, 2, 9)
ys = np.linspace(-2, 2, 9)
X, Y = np.meshgrid(xs, ys)

GX = 2 * X + Y       # partial w.r.t. x
GY = 4 * Y + X       # partial w.r.t. y
mag = np.sqrt(GX ** 2 + GY ** 2)

# Sample a few specific points and report
for pt in [(1, 1), (0, 0), (-1, 1), (2, -1)]:
    g = grad_f(*pt)
    print(f"∇f{pt} = {g}   ||∇f|| = {np.linalg.norm(g):.3f}")

print(f"\nMagnitudes across the grid range from {mag.min():.2f} (at the minimum) to {mag.max():.2f} (at the corners).")
print("Notice ∇f(0, 0) = (0, 0) — the only flat point. That's the minimum of this convex function.")


## Directional derivative

Partial derivatives measure slope along the coordinate axes. The **directional derivative** asks the same question along *any* direction:

$$
D_{\mathbf{u}} f(\mathbf{x}) = \lim_{h \to 0} \frac{f(\mathbf{x} + h \mathbf{u}) - f(\mathbf{x})}{h}
$$

where `u` is a **unit vector** (`||u|| = 1`) telling you which way to step.

The beautiful identity:

$$
D_{\mathbf{u}} f(\mathbf{x}) = \nabla f(\mathbf{x}) \cdot \mathbf{u}
$$

The directional derivative is *just a dot product* between the gradient and the direction. So **once you know the gradient, you know the slope in every possible direction** — there's no need to recompute anything.

In particular, the partials are special cases: take `u = e_i` (the `i`-th standard basis vector) and the dot product picks out the `i`-th component of the gradient, which is `∂f/∂x_i`. Consistent.


In [ ]:
# At point (1, 1), the gradient is (3, 5). Check directional derivatives.
pt = np.array([1.0, 1.0])
g = grad_f(*pt)
print(f"∇f(1, 1) = {g}")

# Slope in the direction (1, 0): should equal ∂f/∂x = 3
u = np.array([1.0, 0.0])
print(f"slope in direction (1, 0): {g @ u:.4f}  (= ∂f/∂x)")

# Slope in the direction (0, 1): should equal ∂f/∂y = 5
u = np.array([0.0, 1.0])
print(f"slope in direction (0, 1): {g @ u:.4f}  (= ∂f/∂y)")

# Slope at 45 degrees, direction (1, 1)/√2
u = np.array([1.0, 1.0]) / np.sqrt(2)
predicted = g @ u
numerical = (f(pt[0] + 1e-5 * u[0], pt[1] + 1e-5 * u[1]) - f(*pt)) / 1e-5
print(f"slope in direction (1,1)/√2: predicted {predicted:.4f}, numerical {numerical:.4f}")


## Steepest ascent — why gradient descent uses -∇f

Given the identity `D_u f = ∇f · u`, ask: *over all unit vectors `u`, which one makes this dot product largest?* From notebook 2 of the linear algebra repo:

$$
\nabla f \cdot \mathbf{u} = \|\nabla f\| \, \|\mathbf{u}\| \, \cos\theta = \|\nabla f\| \cos\theta
$$

(since `||u|| = 1`). The maximum over `θ` is achieved at `θ = 0` — meaning `u` points in the same direction as `∇f`. So:

> **The gradient `∇f` points in the direction of steepest ascent. Its magnitude is the slope in that direction.**

The minimum is at `θ = π`, where `u` points opposite to `∇f`. That direction is the **steepest descent** — and its slope is `-||∇f||`. *This is the whole reason gradient descent works.* If you want to make a loss `L` go down as fast as possible, take a small step in the direction `-∇L`.

The algorithm in one line:

$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \, \nabla L(\mathbf{w}_t)
$$

`α` is the **learning rate**. That's the entire backbone of training every neural network on Earth. Notebook 7 unpacks the variants (momentum, Adam, etc.), but they all sit on top of this one update.


## Level sets and orthogonality

A **level set** of `f` is the set of all `x` where `f(x)` equals some constant `c`:

$$
\{\mathbf{x} \;|\; f(\mathbf{x}) = c\}
$$

In 2D, level sets are **contour lines** (think topographic maps). In 3D they are surfaces. They are the lines on which `f` is *flat* — no change as you walk along them.

The clean geometric fact:

> **The gradient `∇f` is orthogonal to the level set passing through `x`.**

Quick proof. Walk along the level set. By definition, `f` doesn't change, so the directional derivative along the level-set direction is zero: `∇f · v = 0`. That's exactly the definition of orthogonality.

Two ML payoffs of this fact:

- **Loss landscapes.** Contour plots of `L(w)` show you the geometry of training. The gradient is always perpendicular to the contour you're standing on, pointing toward higher loss. Gradient descent takes you orthogonal to contours, cutting straight toward the minimum.
- **Constrained optimization (Lagrange multipliers).** If you minimize `f` subject to `g(x) = 0`, the optimum is where `∇f` is parallel to `∇g`. That's because both must be orthogonal to the constraint surface, and parallel gradients mean their directions of steepest ascent line up. This is the seed of SVMs and many regularization derivations.


## Worked example — gradient of MSE for linear regression

This is the calculation that runs inside every linear regression library. With data matrix `X ∈ R^{n × d}`, target vector `y ∈ R^n`, and weights `w ∈ R^d`, the MSE loss is:

$$
L(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \mathbf{x}_i \cdot \mathbf{w})^2 = \frac{1}{n} \|\mathbf{y} - X\mathbf{w}\|^2
$$

We want `∇_w L`. Expanding the squared norm:

$$
L(\mathbf{w}) = \frac{1}{n} (\mathbf{y} - X\mathbf{w})^T (\mathbf{y} - X\mathbf{w}) = \frac{1}{n} \left(\mathbf{y}^T \mathbf{y} - 2 \mathbf{w}^T X^T \mathbf{y} + \mathbf{w}^T X^T X \mathbf{w}\right)
$$

Differentiate each piece (we'll use two matrix-calculus identities from notebook 5, but they're easy to anticipate):

- `∇_w (\mathbf{w}^T X^T \mathbf{y}) = X^T \mathbf{y}`  — gradient of a linear form in `w` is the constant vector.
- `∇_w (\mathbf{w}^T X^T X \mathbf{w}) = 2 X^T X \mathbf{w}`  — gradient of a quadratic form with symmetric matrix is `2 A w`.

Combining:

$$
\nabla_{\mathbf{w}} L = \frac{2}{n} \left(X^T X \mathbf{w} - X^T \mathbf{y}\right) = \frac{2}{n} X^T (X \mathbf{w} - \mathbf{y})
$$

Setting `∇L = 0` recovers the **normal equations** from the linear algebra repo. Or, if we step iteratively along `-∇L`, that's **gradient descent for linear regression** — the simplest possible training loop.

Numerical verification next cell.


In [ ]:
# Verify the MSE gradient formula via finite differences
rng = np.random.default_rng(0)
n, d = 30, 4
X = rng.normal(size=(n, d))
y = rng.normal(size=n)
w = rng.normal(size=d)

def mse_loss(w):
    return np.mean((y - X @ w) ** 2)

# Analytical gradient
grad_analytic = (2 / n) * X.T @ (X @ w - y)

# Numerical gradient — central difference along each coordinate
def numerical_grad(L, w, h=1e-5):
    g = np.zeros_like(w)
    for i in range(len(w)):
        w_plus, w_minus = w.copy(), w.copy()
        w_plus[i]  += h
        w_minus[i] -= h
        g[i] = (L(w_plus) - L(w_minus)) / (2 * h)
    return g

grad_numerical = numerical_grad(mse_loss, w)
print("analytical  :", grad_analytic.round(6))
print("numerical   :", grad_numerical.round(6))
print("max abs diff:", np.max(np.abs(grad_analytic - grad_numerical)))

# One gradient descent step shows the loss drops
alpha = 0.05
w_new = w - alpha * grad_analytic
print(f"\nloss before step: {mse_loss(w):.4f}")
print(f"loss after  step: {mse_loss(w_new):.4f}")


## Where this appears in ML

Gradients are the single most-used object in machine learning. Every backward pass produces one; every optimizer consumes one.

- **Gradient descent.** Every training loop is `w ← w - α ∇L`. The "directional derivative is maximized along the gradient direction" fact is the whole reason this works.
- **Stochastic gradient descent (SGD).** Same update, but `∇L` is approximated using a mini-batch of samples instead of the full dataset. Way cheaper, and the noise turns out to help escape saddle points.
- **Vanishing / exploding gradients.** When `||∇L||` is near zero, training stalls. When it explodes, weights blow up. Gradient clipping (`g ← g · min(1, threshold/||g||)`) caps the magnitude.
- **Saliency maps & integrated gradients.** Use `∂L/∂x` (gradient w.r.t. inputs, not weights) to explain which pixels matter for a prediction.
- **Adversarial examples.** `x_adv = x + ε · sign(∇_x L)` — perturb the input in the direction that *increases* the loss most. The clearest demonstration of "the gradient points uphill."
- **Lagrangians and constrained optimization.** SVMs, normalizing flows with constraints, and lots of physics-informed neural nets use Lagrange multipliers, which leans on the gradient-perpendicular-to-level-set fact.
- **Policy gradients in RL.** `∇_θ J(θ) = E[∇_θ log π_θ(a|s) · R]` — the policy gradient theorem is *literally* a multivariable derivative.
- **Implicit differentiation.** Sometimes you want gradients through optimization itself (e.g., differentiable convex layers, meta-learning). The implicit function theorem extends the partial-derivative apparatus to these cases.

Next notebook: **the chain rule and backpropagation** — we lift the gradient from "what's the slope of `L(w)`?" to "how do I efficiently compute the gradient when `L` is `100` layers deep?" The chain rule is the answer, and backprop is the algorithm.
